# Migracao Oracle → ClickHouse

**IMPORTANTE**: Feche o Jupyter completamente e execute no terminal:
```bash
pip install "numpy<2" pandas --force-reinstall
```
Depois abra o Jupyter novamente.

## 1. Verificar NumPy

In [67]:
import numpy as np
print(f"NumPy: {np.__version__}")
# Deve mostrar 1.x.x

NumPy: 1.26.4


## 2. Criar Spark Session (COM Arrow desabilitado)

In [68]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("oracle-clickhouse-migration")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print("Spark Session criada (Arrow desabilitado)")

Spark Session criada (Arrow desabilitado)


In [69]:
## 3. Configurar ClickHouse JDBC


clickhouse_host = "e1a1lieug8.us-central1.gcp.clickhouse.cloud"
clickhouse_port = 8443
clickhouse_user = "default"
clickhouse_password = "_uv765EvWphL_"
clickhouse_database = "default"

# JDBC URL para ClickHouse
clickhouse_jdbc_url = f"jdbc:clickhouse://{clickhouse_host}:{clickhouse_port}/{clickhouse_database}?ssl=true"

print(f"ClickHouse JDBC URL: {clickhouse_jdbc_url}")

# Configurações JDBC para ClickHouse
clickhouse_jdbc_props = {
    "user": clickhouse_user,
    "password": clickhouse_password,
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
    "ssl": "true",
    "sslmode": "strict"
}

print("ClickHouse configurado para uso com Spark JDBC")

ClickHouse JDBC URL: jdbc:clickhouse://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443/default?ssl=true
ClickHouse configurado para uso com Spark JDBC


## 4. Configurar Oracle


In [70]:
oracle_host = "10.255.150.11"
oracle_port = 1521
oracle_service = "bi.grupotracker.com.br"
oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
print(f"JDBC URL: {jdbc_url}")

JDBC URL: jdbc:oracle:thin:@//10.255.150.11:1521/bi.grupotracker.com.br


## 5. Teste de Conexão Oracle


In [71]:
df_test = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", oracle_user)
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("query", "SELECT 1 AS ok FROM dual")
    .load()
)

df_test.show()
print("[OK] Oracle funcionando")

+------------+
|          OK|
+------------+
|1.0000000000|
+------------+

[OK] Oracle funcionando


## 6. Definir Tabelas para Migração


In [64]:
tables_to_extract = [
    "ginf.depara_cliente",
    "ginf.BASE_CEP_COMPLETA",
    "ginf.TST_CONTRATOS_BI",
    "siga.SC5030",
    "siga.SC6030",
    "ginf.TST_HISTORICO_SOLICITACOES",
    "ginf.TST_SOLICIT_CADASTRADAS",
    "siga.SD2030",
    "siga.SF2030",
    "siga.ZTX030",
    "ginf.TST_CONTRATOS",
]

print(f"Total: {len(tables_to_extract)} tabelas")
for t in tables_to_extract:
    print(f"  - {t}")

Total: 11 tabelas
  - ginf.depara_cliente
  - ginf.BASE_CEP_COMPLETA
  - ginf.TST_CONTRATOS_BI
  - siga.SC5030
  - siga.SC6030
  - ginf.TST_HISTORICO_SOLICITACOES
  - ginf.TST_SOLICIT_CADASTRADAS
  - siga.SD2030
  - siga.SF2030
  - siga.ZTX030
  - ginf.TST_CONTRATOS


## 7. Migração Inicial - RAW/Bronze Layer

In [65]:


import clickhouse_connect
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Manter cliente ClickHouse para gerenciar progresso
client = clickhouse_connect.get_client(
    host=clickhouse_host,
    port=8443,
    username=clickhouse_user,
    password=clickhouse_password,
    database=clickhouse_database,
    secure=True,
    connect_timeout=60,
    send_receive_timeout=300
)
print("ClickHouse client OK (para gerenciar progresso)\n")

# Limite de linhas por batch
MAX_ROWS = 50000

# Criar tabela de progresso no ClickHouse
try:
    client.command("""
        CREATE TABLE IF NOT EXISTS migration_progress
        (
            oracle_table String,
            ch_table String,
            rows_collected Int64,
            total_rows Float64,
            rows_remaining Float64,
            last_id String,
            id_column String,
            status String,
            error String,
            updated_at DateTime DEFAULT now()
        ) ENGINE = ReplacingMergeTree(updated_at)
        ORDER BY oracle_table
    """)
    print("[INFO] Tabela de progresso criada/verificada\n")
except Exception as e:
    print(f"[AVISO] Erro ao criar tabela de progresso: {str(e)[:100]}\n")


def spark_to_ch_type(spark_type):
    """Converte tipo Spark para tipo ClickHouse"""
    tipo = str(spark_type).lower()
    if 'string' in tipo or 'varchar' in tipo:
        base = 'String'
    elif 'decimal' in tipo or 'number' in tipo:
        base = 'String'
    elif 'int' in tipo:
        base = 'Int64'
    elif 'long' in tipo or 'bigint' in tipo:
        base = 'Int64'
    elif 'double' in tipo or 'float' in tipo:
        base = 'Float64'
    elif 'date' in tipo:
        base = 'String'
    elif 'timestamp' in tipo:
        base = 'String'
    elif 'binary' in tipo:
        base = 'String'
    else:
        base = 'String'
    return f'Nullable({base})'


def create_clickhouse_table_from_spark(df, ch_tbl):
    """Cria tabela no ClickHouse baseada no schema do DataFrame Spark"""
    ch_cols = []
    for field in df.schema.fields:
        ch_type = spark_to_ch_type(field.dataType)
        ch_cols.append(f"`{field.name}` {ch_type}")

    create_sql = f"""
        CREATE TABLE IF NOT EXISTS {ch_tbl} (
            {', '.join(ch_cols)}
        ) ENGINE = MergeTree()
        ORDER BY tuple()
    """

    client.command(create_sql)
    return True


def remove_duplicates_spark(df, id_col):
    """
    [RAW/BRONZE LAYER] Mantém dados como estão - SEM REMOÇÃO DE DUPLICATAS

    Na arquitetura Medallion:
    - BRONZE/RAW: Dados brutos do Oracle, exatamente como estão (incluindo duplicatas)
    - SILVER: Dados limpos e transformados (onde a remoção de duplicatas deve ocorrer)
    - GOLD: Dados agregados e prontos para análise

    Esta função retorna o DataFrame original sem modificações.
    """
    print("  [RAW] Dados mantidos como estão (incluindo duplicatas)")
    return df

    # ---- CÓDIGO COMENTADO PARA USO NA SILVER LAYER ----
    # Para remover duplicatas na camada SILVER, descomente o código abaixo:
    #
    if not id_col or id_col not in df.columns:
        # Se não houver coluna ID, usar distinct() em todas as colunas
        original_count = df.count()
        df_clean = df.distinct()
        clean_count = df_clean.count()
        if original_count > clean_count:
            print(f"  [SILVER] {original_count - clean_count} duplicata(s) exata(s) removida(s)")
        return df_clean

    # Usar window function para manter apenas a primeira ocorrência de cada ID
    window_spec = Window.partitionBy(id_col).orderBy(F.monotonically_increasing_id())
    df_dedup = df.withColumn("row_num", F.row_number().over(window_spec)) \
                 .filter(F.col("row_num") == 1) \
                 .drop("row_num")

    original_count = df.count()
    dedup_count = df_dedup.count()

    if original_count > dedup_count:
        print(f"  [SILVER] {original_count - dedup_count} duplicata(s) removida(s)")

    return df_dedup


def save_progress_to_ch(oracle_tbl, ch_tbl, stats):
    """Salva progresso no ClickHouse usando pandas"""
    try:
        progress_data = pd.DataFrame([{
            'oracle_table': oracle_tbl,
            'ch_table': ch_tbl,
            'rows_collected': int(stats.get('rows_collected', 0)),
            'total_rows': float(stats.get('total_rows', 0)),
            'rows_remaining': float(stats.get('rows_remaining', 0)),
            'last_id': str(stats.get('last_id', '')),
            'id_column': str(stats.get('id_column', '')),
            'status': stats.get('status', 'unknown'),
            'error': str(stats.get('error', ''))[:500]
        }])
        client.insert_df('migration_progress', progress_data)
    except Exception as e:
        print(f"  [AVISO] Erro ao salvar progresso: {str(e)[:80]}")


sep = "=" * 60
print(sep)
print(f"MIGRAÇÃO ORACLE → CLICKHOUSE VIA SPARK")
print(f"Batch size: {MAX_ROWS:,} linhas")
print(sep)

start = datetime.now()
ok = 0
fail = 0
total_rows = 0
failed = []
migration_stats = {}

# Mapear tabelas
ch_tables = {
    "depara_cliente": "ginf.depara_cliente",
    "base_cep_completa": "ginf.BASE_CEP_COMPLETA",
    "bistage": "bistage.TST_CONTRATOS_BI",
    "sc5030": "siga.SC5030",
    "sc6030": "siga.SC6030",
    "tst_historico_solicitacoes": "ginf.TST_HISTORICO_SOLICITACOES",
    "tst_solicit_cadastradas": "ginf.TST_SOLICIT_CADASTRADAS",
    "sd2030": "siga.SD2030",
    "sf2030": "siga.SF2030",
    "ztx030": "siga.ZTX030",
    "tst_contratos": "ginf.TST_CONTRATOS"
}

for i, (ch_tbl, oracle_tbl) in enumerate(ch_tables.items(), 1):
    pct = (i / len(ch_tables)) * 100
    print(f"\n[{i}/{len(ch_tables)}] ({pct:.1f}%) {oracle_tbl} -> {ch_tbl}")

    try:
        t0 = datetime.now()

        # 1. CONTAR linhas no Oracle
        try:
            df_count = spark.read.format("jdbc") \
                .options(**jdbc_opts) \
                .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {oracle_tbl}) tmp") \
                .load()
            total_oracle_rows = float(df_count.collect()[0]['CNT'])
            print(f"  Total no Oracle: {total_oracle_rows:,.0f} linhas")
        except Exception as count_err:
            print(f"  [ERRO] Falha ao contar: {str(count_err)[:100]}")
            stats = {
                "status": "failed",
                "error": f"Count failed: {str(count_err)[:100]}",
                "rows_collected": 0,
                "total_rows": 0,
                "rows_remaining": 0,
                "last_id": "",
                "id_column": ""
            }
            migration_stats[oracle_tbl] = stats
            save_progress_to_ch(oracle_tbl, ch_tbl, stats)
            fail += 1
            failed.append(oracle_tbl)
            continue

        # 2. LER dados do Oracle com ORDER BY
        print(f"  Lendo até {MAX_ROWS:,} linhas do Oracle...")
        df = spark.read.format("jdbc") \
            .options(**jdbc_opts) \
            .option("dbtable", f"(SELECT * FROM {oracle_tbl} WHERE ROWNUM <= {MAX_ROWS}) tmp") \
            .load()

        nrows = df.count()

        if nrows == 0:
            print(f"  [AVISO] Tabela vazia")
            stats = {
                "rows_collected": 0,
                "total_rows": total_oracle_rows,
                "rows_remaining": total_oracle_rows,
                "last_id": "",
                "id_column": "",
                "status": "empty",
                "error": ""
            }
            migration_stats[oracle_tbl] = stats
            save_progress_to_ch(oracle_tbl, ch_tbl, stats)
            fail += 1
            failed.append(oracle_tbl)
            continue

        print(f"  Lidas: {nrows:,} linhas")

        # 3. IDENTIFICAR coluna ID
        cols = df.columns
        id_col = None
        for col in cols:
            if 'ID' in col.upper() or col == cols[0]:
                id_col = col
                break

        print(f"  Coluna ID: {id_col}")

        # 4. [RAW LAYER] Manter dados como estão (incluindo duplicatas)
        df_clean = df
        nrows_clean = df_clean.count()

        # 5. CRIAR tabela no ClickHouse
        try:
            # Testar se tabela existe
            client.command(f"SELECT 1 FROM {ch_tbl} LIMIT 1")
            print(f"  Tabela '{ch_tbl}' já existe")
        except:
            print(f"  Criando tabela '{ch_tbl}' no ClickHouse...")
            create_clickhouse_table_from_spark(df_clean, ch_tbl)
            print(f"  [OK] Tabela criada")

        # 6. ESCREVER no ClickHouse usando Spark JDBC
        print(f"  Gravando {nrows_clean:,} linhas no ClickHouse via Spark...")

        df_clean.write \
            .format("jdbc") \
            .option("url", clickhouse_jdbc_url) \
            .option("dbtable", ch_tbl) \
            .option("user", clickhouse_user) \
            .option("password", clickhouse_password) \
            .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
            .option("batchsize", 10000) \
            .option("isolationLevel", "NONE") \
            .mode("append") \
            .save()

        # 7. CAPTURAR último ID
        last_id = None
        if id_col:
            last_row = df_clean.orderBy(F.col(id_col).desc()).first()
            last_id = last_row[id_col] if last_row else None

        dur = (datetime.now() - t0).total_seconds()
        rows_remaining = max(0, total_oracle_rows - nrows_clean)

        print(f"  [OK] {nrows_clean:,} linhas inseridas em {dur:.2f}s")
        print(f"  Restante: {rows_remaining:,.0f} linhas")
        if last_id:
            print(f"  Último ID: {last_id}")

        # 8. SALVAR estatísticas
        stats = {
            "rows_collected": nrows_clean,
            "total_rows": total_oracle_rows,
            "rows_remaining": rows_remaining,
            "last_id": str(last_id) if last_id else "",
            "id_column": id_col or "",
            "status": "complete" if rows_remaining == 0 else "partial",
            "error": ""
        }
        migration_stats[oracle_tbl] = stats
        save_progress_to_ch(oracle_tbl, ch_tbl, stats)

        ok += 1
        total_rows += nrows_clean

    except Exception as e:
        msg = str(e)[:200]
        print(f"  [ERRO] {msg}")
        fail += 1
        failed.append(oracle_tbl)
        stats = {
            "status": "failed",
            "error": msg,
            "rows_collected": 0,
            "total_rows": 0,
            "rows_remaining": 0,
            "last_id": "",
            "id_column": ""
        }
        migration_stats[oracle_tbl] = stats
        save_progress_to_ch(oracle_tbl, ch_tbl, stats)

dur = (datetime.now() - start).total_seconds()

print(f"\n{sep}")
print("RESUMO")
print(sep)
print(f"Sucesso: {ok}/{len(ch_tables)}")
print(f"Falhas: {fail}")
print(f"Total linhas coletadas: {total_rows:,}")
print(f"Tempo: {dur:.2f}s")

if failed:
    print(f"\nTabelas com erro:")
    for t in failed:
        print(f"  - {t}")

print(f"\n{sep}")
print("[INFO] Progresso salvo no migration_progress")
print(sep)

ClickHouse client OK (para gerenciar progresso)

[INFO] Tabela de progresso criada/verificada

MIGRAÇÃO ORACLE → CLICKHOUSE VIA SPARK
Batch size: 50,000 linhas

[1/11] (9.1%) ginf.depara_cliente -> depara_cliente
  Total no Oracle: 47 linhas
  Lendo até 50,000 linhas do Oracle...
  Lidas: 47 linhas
  Coluna ID: DOC_CLIENTE
  Tabela 'depara_cliente' já existe
  Gravando 47 linhas no ClickHouse via Spark...
  [OK] 47 linhas inseridas em 4.83s
  Restante: 0 linhas
  Último ID: 88473731000103

[2/11] (18.2%) ginf.BASE_CEP_COMPLETA -> base_cep_completa
  Total no Oracle: 96,778 linhas
  Lendo até 50,000 linhas do Oracle...
  Lidas: 50,000 linhas
  Coluna ID: FAIXA_CEP_CORREIO
  Tabela 'base_cep_completa' já existe
  Gravando 50,000 linhas no ClickHouse via Spark...
  [OK] 50,000 linhas inseridas em 10.41s
  Restante: 46,778 linhas
  Último ID: 99969

[3/11] (27.3%) bistage.TST_CONTRATOS_BI -> bistage
  Total no Oracle: 3,047,626 linhas
  Lendo até 50,000 linhas do Oracle...
  Lidas: 50,000 

## 8. Migração Incremental - Continuar de onde parou

In [78]:
## 8. Migração Incremental - Continuar de onde parou

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from datetime import datetime
import time

print("=" * 60)
print("MIGRAÇÃO INCREMENTAL COMPLETA - ORACLE → CLICKHOUSE")
print("=" * 60)
MAX_ROWS = 1000000
# Configurações de performance
MAX_ITERATIONS = 1000  # Máximo de iterações para evitar loop infinito
SLEEP_BETWEEN_BATCHES = 2  # Segundos entre batches (evitar sobrecarga)

iteration = 0
total_migrated = 0

while iteration < MAX_ITERATIONS:
    iteration += 1
    
    print(f"\n{'='*60}")
    print(f"ITERAÇÃO {iteration}/{MAX_ITERATIONS}")
    print(f"{'='*60}")
    
    # Obter progresso do ClickHouse
    progress_query = """
    SELECT 
        oracle_table,
        ch_table,
        rows_collected,
        total_rows,
        rows_remaining,
        last_id,
        id_column,
        status,
        error
    FROM migration_progress
    FINAL
    WHERE status = 'partial'
    ORDER BY rows_remaining DESC  -- Priorizar tabelas maiores
    """
    
    progress_df = client.query_df(progress_query)
    
    if len(progress_df) == 0:
        print("\n" + "="*60)
        print("✅ MIGRAÇÃO COMPLETA - TODAS AS TABELAS FINALIZADAS!")
        print("="*60)
        break
    
    print(f"[INFO] {len(progress_df)} tabela(s) pendente(s)\n")
    
    batch_inserted = 0
    
    for idx, row in progress_df.iterrows():
        oracle_tbl = row['oracle_table']
        ch_tbl = row['ch_table']
        last_id = row['last_id']
        id_col = row['id_column']
        rows_collected = int(row['rows_collected'])
        rows_remaining = float(row['rows_remaining'])
        
        print(f"\n[{idx + 1}/{len(progress_df)}] {oracle_tbl} -> {ch_tbl}")
        print(f"  Progresso: {rows_collected:,} / {rows_collected + rows_remaining:,.0f} linhas")
        print(f"  Restante: {rows_remaining:,.0f} linhas")
        
        try:
            # Validação pré-inserção
            try:
                df_ch_count = spark.read.format("jdbc") \
                    .option("url", clickhouse_jdbc_url) \
                    .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {ch_tbl}) tmp") \
                    .option("user", clickhouse_user) \
                    .option("password", clickhouse_password) \
                    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
                    .load()
                
                ch_count = int(df_ch_count.collect()[0]['cnt'])
                
                if ch_count != rows_collected:
                    print(f"  [AVISO] Ajustando contador: {ch_count:,} linhas no ClickHouse")
                    rows_collected = ch_count
                    
            except Exception as validate_err:
                print(f"  [AVISO] Validação não disponível: {str(validate_err)[:60]}")
            
            t0 = datetime.now()
            
            # Construir query Oracle com paginação correta
            if id_col and last_id:
                query = f"""
                (SELECT * FROM (
                    SELECT * FROM {oracle_tbl} 
                    WHERE {id_col} > '{last_id}'
                    ORDER BY {id_col}
                ) WHERE ROWNUM <= {MAX_ROWS}) tmp
                """
            else:
                query = f"""
                (SELECT * FROM 
                    (SELECT a.*, ROWNUM rnum FROM 
                        (SELECT * FROM {oracle_tbl} ORDER BY 1) a
                     WHERE ROWNUM <= {rows_collected + MAX_ROWS})
                 WHERE rnum > {rows_collected}) tmp
                """
            
            # Ler batch do Oracle
            df = spark.read.format("jdbc") \
                .options(**jdbc_opts) \
                .option("dbtable", query) \
                .option("fetchsize", 10000) \
                .load()
            
            # Remover coluna ROWNUM se existir
            if 'rnum' in df.columns:
                df = df.drop('rnum')
            
            nrows = df.count()
            
            if nrows == 0:
                print(f"  ✅ Migração COMPLETA para esta tabela!")
                stats_complete = pd.DataFrame([{
                    'oracle_table': oracle_tbl,
                    'ch_table': ch_tbl,
                    'rows_collected': rows_collected,
                    'total_rows': float(rows_collected),
                    'rows_remaining': 0.0,
                    'last_id': last_id,
                    'id_column': id_col,
                    'status': 'complete',
                    'error': ''
                }])
                client.insert_df('migration_progress', stats_complete)
                continue
            
            print(f"  Lidas: {nrows:,} linhas")
            
            # [RAW LAYER] Manter dados como estão
            df_clean = df
            
            # Capturar novo último ID
            new_last_id = last_id
            if id_col and id_col in df_clean.columns:
                last_row = df_clean.orderBy(F.col(id_col).desc()).first()
                new_last_id = last_row[id_col] if last_row else last_id
            
            # Escrever no ClickHouse via Spark JDBC
            print(f"  Gravando {nrows:,} linhas no ClickHouse...")
            
            df_clean.write \
                .format("jdbc") \
                .option("url", clickhouse_jdbc_url) \
                .option("dbtable", ch_tbl) \
                .option("user", clickhouse_user) \
                .option("password", clickhouse_password) \
                .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
                .option("batchsize", 10000) \
                .option("isolationLevel", "NONE") \
                .option("numPartitions", 4) \
                .mode("append") \
                .save()
            
            # Atualizar progresso
            new_total_collected = rows_collected + nrows
            
            # Contar restante
            if id_col and new_last_id:
                df_remaining = spark.read.format("jdbc") \
                    .options(**jdbc_opts) \
                    .option("dbtable", f"(SELECT COUNT(*) as cnt FROM {oracle_tbl} WHERE {id_col} > '{new_last_id}') tmp") \
                    .load()
                rows_remaining = float(df_remaining.collect()[0]['CNT'])
            else:
                rows_remaining = max(0, rows_remaining - nrows)
            
            dur = (datetime.now() - t0).total_seconds()
            print(f"  ✅ {nrows:,} linhas inseridas em {dur:.2f}s")
            print(f"  Total: {new_total_collected:,} | Restante: {rows_remaining:,.0f}")
            
            # Salvar progresso
            new_status = 'complete' if rows_remaining == 0 else 'partial'
            stats_update = pd.DataFrame([{
                'oracle_table': oracle_tbl,
                'ch_table': ch_tbl,
                'rows_collected': new_total_collected,
                'total_rows': float(new_total_collected + rows_remaining),
                'rows_remaining': rows_remaining,
                'last_id': str(new_last_id),
                'id_column': id_col,
                'status': new_status,
                'error': ''
            }])
            client.insert_df('migration_progress', stats_update)
            
            batch_inserted += nrows
            total_migrated += nrows
            
        except Exception as e:
            msg = str(e)[:300]
            print(f"  ❌ ERRO: {msg}")
            stats_error = pd.DataFrame([{
                'oracle_table': oracle_tbl,
                'ch_table': ch_tbl,
                'rows_collected': rows_collected,
                'total_rows': 0.0,
                'rows_remaining': 0.0,
                'last_id': last_id,
                'id_column': id_col,
                'status': 'error',
                'error': msg[:500]
            }])
            client.insert_df('migration_progress', stats_error)
    
    print(f"\n{'='*60}")
    print(f"Iteração {iteration}: {batch_inserted:,} linhas inseridas")
    print(f"Total acumulado: {total_migrated:,} linhas")
    print(f"{'='*60}")
    
    # Aguardar entre batches
    if batch_inserted > 0:
        time.sleep(SLEEP_BETWEEN_BATCHES)

print(f"\n{'='*60}")
print(f"MIGRAÇÃO FINALIZADA")
print(f"Total de linhas migradas: {total_migrated:,}")
print(f"Iterações executadas: {iteration}")
print(f"{'='*60}")

MIGRAÇÃO INCREMENTAL COMPLETA - ORACLE → CLICKHOUSE

ITERAÇÃO 1/1000
[INFO] 1 tabela(s) pendente(s)


[1/1] ginf.TST_HISTORICO_SOLICITACOES -> tst_historico_solicitacoes
  Progresso: 15,650,000 / 49,511,086 linhas
  Restante: 33,861,086 linhas
  [AVISO] Ajustando contador: 16,650,000 linhas no ClickHouse
  Lidas: 1,000,000 linhas
  Gravando 1,000,000 linhas no ClickHouse...
  ✅ 1,000,000 linhas inseridas em 190.01s
  Total: 17,650,000 | Restante: 32,861,085

Iteração 1: 1,000,000 linhas inseridas
Total acumulado: 1,000,000 linhas

ITERAÇÃO 2/1000
[INFO] 1 tabela(s) pendente(s)


[1/1] ginf.TST_HISTORICO_SOLICITACOES -> tst_historico_solicitacoes
  Progresso: 17,650,000 / 50,511,085 linhas
  Restante: 32,861,085 linhas
  [AVISO] Ajustando contador: 16,650,000 linhas no ClickHouse
  Lidas: 1,000,000 linhas


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## 9. Resetar Erros para Retry

In [75]:


import pandas as pd

print("=" * 60)
print("RESETANDO STATUS DE ERRO PARA PARTIAL")
print("=" * 60)

# Buscar tabelas com erro
error_tables = client.query_df("""
    SELECT oracle_table, ch_table, rows_collected, last_id, id_column
    FROM migration_progress FINAL
    WHERE status = 'error'
    ORDER BY oracle_table
""")

if len(error_tables) == 0:
    print("[INFO] Nenhuma tabela com status 'error' encontrada")
else:
    print(f"[INFO] {len(error_tables)} tabela(s) com status 'error'\n")

    for idx, row in error_tables.iterrows():
        oracle_tbl = row['oracle_table']
        ch_tbl = row['ch_table']
        rows_collected = int(row['rows_collected'])
        last_id = row['last_id']
        id_col = row['id_column']

        print(f"  Resetando: {oracle_tbl} ({rows_collected:,} linhas coletadas)")

        # Atualizar status para partial
        reset_data = pd.DataFrame([{
            'oracle_table': oracle_tbl,
            'ch_table': ch_tbl,
            'rows_collected': rows_collected,
            'total_rows': 0.0,  # Será recalculado
            'rows_remaining': 0.0,
            'last_id': last_id,
            'id_column': id_col,
            'status': 'partial',
            'error': ''
        }])

        client.insert_df('migration_progress', reset_data)

    print(f"\n{'='*60}")
    print(f"✅ {len(error_tables)} tabela(s) resetadas para 'partial'")
    print(f"{'='*60}")
    print("\nAgora execute a célula de migração incremental novamente.")

RESETANDO STATUS DE ERRO PARA PARTIAL
[INFO] Nenhuma tabela com status 'error' encontrada


## 10. Funções: Detecção e Remoção de Duplicatas (para Silver Layer)

In [76]:


def check_and_remove_duplicates_in_memory(pdf, id_col=None):
    """
    Verifica duplicatas no DataFrame em memória e remove antes de inserir

    Retorna: (pdf_clean, stats_dict)
    """
    original_count = len(pdf)

    if original_count == 0:
        return pdf, {'duplicates_found': 0, 'duplicates_removed': 0, 'final_count': 0}

    # 1. Duplicatas EXATAS (todas as colunas iguais)
    duplicates_exact = pdf.duplicated(keep='first')
    num_exact_dups = duplicates_exact.sum()

    if num_exact_dups > 0:
        print(f"  [AVISO] {num_exact_dups} linha(s) com DUPLICATA EXATA encontrada(s)")
        # Remove duplicatas exatas, mantendo a primeira ocorrência
        pdf = pdf[~duplicates_exact].copy()

    # 2. Duplicatas por ID (se id_col especificado)
    if id_col and id_col in pdf.columns:
        duplicates_id = pdf.duplicated(subset=[id_col], keep='first')
        num_id_dups = duplicates_id.sum()

        if num_id_dups > 0:
            print(f"  [AVISO] {num_id_dups} linha(s) com ID duplicado ({id_col}) encontrada(s)")
            # Mostra exemplo das duplicatas
            dup_ids = pdf[duplicates_id][id_col].unique()[:5]
            print(f"    Exemplos de IDs duplicados: {list(dup_ids)}")
            # Remove duplicatas por ID, mantendo a primeira
            pdf = pdf[~duplicates_id].copy()

    final_count = len(pdf)
    total_removed = original_count - final_count

    stats = {
        'original_count': original_count,
        'duplicates_exact': num_exact_dups,
        'duplicates_id': num_id_dups if id_col else 0,
        'total_removed': total_removed,
        'final_count': final_count
    }

    if total_removed > 0:
        print(f"  [LIMPEZA] {total_removed} duplicata(s) removida(s)")
        print(f"  [INFO] Linhas finais: {final_count:,} (de {original_count:,})")

    return pdf, stats


def check_duplicates_in_clickhouse(ch_tbl, id_col=None):
    """
    Verifica se existem duplicatas na tabela ClickHouse

    Retorna: dict com estatísticas de duplicatas
    """
    print(f"\n{'='*60}")
    print(f"VERIFICANDO DUPLICATAS EM: {ch_tbl}")
    print(f"{'='*60}")

    try:
        # Contar total de linhas
        total_rows = client.query(f"SELECT count() FROM {ch_tbl}").result_rows[0][0]
        print(f"Total de linhas: {total_rows:,}")

        if total_rows == 0:
            print("  [INFO] Tabela vazia - sem duplicatas")
            return {'has_duplicates': False, 'total_rows': 0}

        # Verificar duplicatas exatas (todas as colunas)
        dup_exact_query = f"""
        SELECT count() as dup_count
        FROM (
            SELECT *, count() as cnt
            FROM {ch_tbl}
            GROUP BY *
            HAVING cnt > 1
        )
        """

        try:
            dup_exact = client.query(dup_exact_query).result_rows[0][0]
            print(f"Linhas com duplicata exata: {dup_exact:,}")
        except:
            dup_exact = 0
            print("  [INFO] Não foi possível verificar duplicatas exatas (pode ser limitação do schema)")

        # Verificar duplicatas por ID (se especificado)
        dup_by_id = 0
        unique_ids = total_rows

        if id_col:
            try:
                # Contar IDs únicos
                unique_ids = client.query(f"SELECT count(DISTINCT {id_col}) FROM {ch_tbl}").result_rows[0][0]
                dup_by_id = total_rows - unique_ids

                if dup_by_id > 0:
                    print(f"\n❌ DUPLICATAS DETECTADAS POR ID ({id_col}):")
                    print(f"   Total de linhas: {total_rows:,}")
                    print(f"   IDs únicos: {unique_ids:,}")
                    print(f"   Linhas duplicadas: {dup_by_id:,}")

                    # Mostrar exemplos de IDs duplicados
                    dup_examples_query = f"""
                    SELECT {id_col}, count() as cnt
                    FROM {ch_tbl}
                    GROUP BY {id_col}
                    HAVING cnt > 1
                    ORDER BY cnt DESC
                    LIMIT 10
                    """
                    dup_examples = client.query(dup_examples_query).result_rows

                    print(f"\n   Top 10 IDs mais duplicados:")
                    for id_val, cnt in dup_examples:
                        print(f"     - {id_val}: {cnt} ocorrências")
                else:
                    print(f"\n✅ SEM DUPLICATAS por ID ({id_col})")
                    print(f"   Todas as {total_rows:,} linhas têm IDs únicos")

            except Exception as e:
                print(f"  [AVISO] Erro ao verificar duplicatas por ID: {str(e)[:100]}")

        has_dups = (dup_exact > 0) or (dup_by_id > 0)

        return {
            'has_duplicates': has_dups,
            'total_rows': total_rows,
            'unique_ids': unique_ids,
            'duplicates_exact': dup_exact,
            'duplicates_by_id': dup_by_id
        }

    except Exception as e:
        print(f"  [ERRO] Falha ao verificar duplicatas: {str(e)[:200]}")
        return {'has_duplicates': None, 'error': str(e)}


def deduplicate_clickhouse_table(ch_tbl, id_col):
    """
    Remove duplicatas de uma tabela ClickHouse criando versão limpa
    """
    print(f"\n{'='*60}")
    print(f"REMOVENDO DUPLICATAS DE: {ch_tbl}")
    print(f"{'='*60}")

    try:
        # Verificar se existe duplicata
        stats = check_duplicates_in_clickhouse(ch_tbl, id_col)

        if not stats.get('has_duplicates'):
            print("  [INFO] Nenhuma duplicata encontrada - tabela já está limpa")
            return

        temp_table = f"{ch_tbl}_temp_dedup"

        print(f"\n[1/4] Criando tabela temporária sem duplicatas...")

        # Criar tabela temporária com dados únicos (usa DISTINCT ON ou ROW_NUMBER)
        dedup_query = f"""
        CREATE TABLE {temp_table} ENGINE = MergeTree() ORDER BY tuple()
        AS SELECT * FROM {ch_tbl}
        GROUP BY {id_col}, *
        HAVING count() = 1 OR rowNumberInBlock() = 1
        """

        # Alternativa mais segura: usar DISTINCT apenas por ID, mantendo primeira ocorrência
        dedup_query_safe = f"""
        CREATE TABLE {temp_table} ENGINE = MergeTree() ORDER BY tuple()
        AS SELECT * FROM (
            SELECT *, row_number() OVER (PARTITION BY {id_col}) as rn
            FROM {ch_tbl}
        )
        WHERE rn = 1
        """

        try:
            client.command(dedup_query_safe)
        except:
            # Fallback: usar GROUP BY simples
            client.command(f"""
                CREATE TABLE {temp_table} ENGINE = MergeTree() ORDER BY tuple()
                AS SELECT * FROM {ch_tbl}
                GROUP BY {id_col}
            """)

        print(f"[2/4] Contando linhas na tabela limpa...")
        new_count = client.query(f"SELECT count() FROM {temp_table}").result_rows[0][0]
        old_count = stats['total_rows']
        removed = old_count - new_count

        print(f"  Antes: {old_count:,} linhas")
        print(f"  Depois: {new_count:,} linhas")
        print(f"  Removidas: {removed:,} duplicatas")

        print(f"\n[3/4] Substituindo tabela original...")
        client.command(f"DROP TABLE {ch_tbl}")
        client.command(f"RENAME TABLE {temp_table} TO {ch_tbl}")

        print(f"[4/4] Atualizando progresso...")
        # Atualizar rows_collected no migration_progress
        client.command(f"""
            INSERT INTO migration_progress
            SELECT
                oracle_table,
                ch_table,
                {new_count} as rows_collected,
                total_rows,
                rows_remaining,
                last_id,
                id_column,
                'partial' as status,
                'Deduplicada' as error
            FROM migration_progress FINAL
            WHERE ch_table = '{ch_tbl}'
            LIMIT 1
        """)

        print(f"Deduplicação concluída com sucesso!")

    except Exception as e:
        print(f"ERRO ao deduplificar: {str(e)[:200]}")


# EXEMPLO DE USO - Verificar todas as tabelas
print("=" * 60)
print("VERIFICAÇÃO DE DUPLICATAS - TODAS AS TABELAS")
print("=" * 60)

progress = client.query_df("""
    SELECT ch_table, id_column
    FROM migration_progress FINAL
    WHERE status IN ('partial', 'complete')
    ORDER BY ch_table
""")

for idx, row in progress.iterrows():
    ch_tbl = row['ch_table']
    id_col = row['id_column']

    stats = check_duplicates_in_clickhouse(ch_tbl, id_col)

    if stats.get('has_duplicates'):
        print(f"\n⚠️  Para corrigir, execute: deduplicate_clickhouse_table('{ch_tbl}', '{id_col}')")

print("\n" + "=" * 60)

VERIFICAÇÃO DE DUPLICATAS - TODAS AS TABELAS

VERIFICANDO DUPLICATAS EM: base_cep_completa
Total de linhas: 100,030
Linhas com duplicata exata: 50,000

❌ DUPLICATAS DETECTADAS POR ID (FAIXA_CEP_CORREIO):
   Total de linhas: 100,030
   IDs únicos: 50,030
   Linhas duplicadas: 50,000

   Top 10 IDs mais duplicados:
     - 76452: 2 ocorrências
     - 85593: 2 ocorrências
     - 66007: 2 ocorrências
     - 55933: 2 ocorrências
     - 40583: 2 ocorrências
     - 41934: 2 ocorrências
     - 84924: 2 ocorrências
     - 22095: 2 ocorrências
     - 27870: 2 ocorrências
     - 25787: 2 ocorrências

⚠️  Para corrigir, execute: deduplicate_clickhouse_table('base_cep_completa', 'FAIXA_CEP_CORREIO')

VERIFICANDO DUPLICATAS EM: bistage
Total de linhas: 3,059,524
  [INFO] Não foi possível verificar duplicatas exatas (pode ser limitação do schema)

❌ DUPLICATAS DETECTADAS POR ID (NUMERO_CONTRATO):
   Total de linhas: 3,059,524
   IDs únicos: 275,654
   Linhas duplicadas: 2,783,870

   Top 10 IDs mais d

KeyboardInterrupt: 

## 11. Funções: Validação e Diagnóstico

In [77]:


def validate_migration_integrity():
    """
    Valida integridade da migração comparando ClickHouse com progresso registrado
    """
    print("=" * 60)
    print("VALIDAÇÃO DE INTEGRIDADE")
    print("=" * 60)

    progress = client.query_df("""
        SELECT oracle_table, ch_table, rows_collected, status
        FROM migration_progress FINAL
        ORDER BY oracle_table
    """)

    issues = []
    for _, row in progress.iterrows():
        ch_tbl = row['ch_table']
        expected = int(row['rows_collected'])

        try:
            actual = client.query(f"SELECT count() FROM {ch_tbl}").result_rows[0][0]

            if actual != expected:
                issues.append({
                    'table': ch_tbl,
                    'expected': expected,
                    'actual': actual,
                    'diff': actual - expected
                })
                print(f"❌ {ch_tbl}: esperado {expected:,}, atual {actual:,} (diff: {actual - expected:+,})")
            else:
                print(f"✅ {ch_tbl}: {actual:,} linhas OK")
        except Exception as e:
            print(f"⚠️  {ch_tbl}: erro ao consultar - {str(e)[:50]}")

    print("\n" + "=" * 60)
    if not issues:
        print("✅ TODAS AS TABELAS ESTÃO CONSISTENTES")
    else:
        print(f"❌ {len(issues)} TABELA(S) COM INCONSISTÊNCIA")
        print("\nPara corrigir, execute: fix_table_inconsistency('nome_tabela')")
    print("=" * 60)

    return issues


def fix_table_inconsistency(ch_tbl):
    """
    Corrige inconsistência ajustando rows_collected para refletir realidade do ClickHouse
    """
    print(f"Corrigindo inconsistência em: {ch_tbl}")

    # Contar linhas reais no ClickHouse
    actual_count = client.query(f"SELECT count() FROM {ch_tbl}").result_rows[0][0]
    print(f"  Linhas reais no ClickHouse: {actual_count:,}")

    # Obter último ID real da tabela
    progress = client.query_df(f"""
        SELECT oracle_table, id_column
        FROM migration_progress FINAL
        WHERE ch_table = '{ch_tbl}'
        LIMIT 1
    """)

    if len(progress) == 0:
        print(f"  [ERRO] Tabela {ch_tbl} não encontrada no progresso")
        return

    oracle_tbl = progress.iloc[0]['oracle_table']
    id_col = progress.iloc[0]['id_column']

    # Obter último ID do ClickHouse
    last_id_query = f"SELECT max({id_col}) as max_id FROM {ch_tbl}"
    last_id = client.query(last_id_query).result_rows[0][0]

    print(f"  Último ID no ClickHouse: {last_id}")
    print(f"  Atualizando progresso...")

    # Atualizar progresso
    stats_fix = pd.DataFrame([{
        'oracle_table': oracle_tbl,
        'ch_table': ch_tbl,
        'rows_collected': actual_count,
        'total_rows': 0.0,  # Será recalculado na próxima execução
        'rows_remaining': 0.0,
        'last_id': str(last_id) if last_id else '',
        'id_column': id_col,
        'status': 'partial',
        'error': 'Fixed inconsistency'
    }])
    client.insert_df('migration_progress', stats_fix)

    print(f"  ✅ Progresso corrigido para {actual_count:,} linhas")


def show_migration_status():
    """
    Mostra resumo do status atual da migração
    """
    print("=" * 60)
    print("STATUS DA MIGRAÇÃO")
    print("=" * 60)

    progress = client.query_df("""
        SELECT
            oracle_table,
            ch_table,
            rows_collected,
            total_rows,
            rows_remaining,
            status,
            last_id,
            error
        FROM migration_progress FINAL
        ORDER BY
            CASE status
                WHEN 'partial' THEN 1
                WHEN 'error' THEN 2
                WHEN 'complete' THEN 3
                ELSE 4
            END,
            oracle_table
    """)

    for _, row in progress.iterrows():
        status_icon = {
            'partial': '🔄',
            'complete': '✅',
            'error': '❌',
            'failed': '❌'
        }.get(row['status'], '❓')

        print(f"\n{status_icon} {row['oracle_table']} -> {row['ch_table']}")
        print(f"   Status: {row['status'].upper()}")
        print(f"   Coletado: {int(row['rows_collected']):,} / {float(row['total_rows']):,.0f}")

        if row['total_rows'] > 0:
            pct = (row['rows_collected'] / row['total_rows']) * 100
            print(f"   Progresso: {pct:.1f}%")

        if row['status'] == 'partial':
            print(f"   Último ID: {row['last_id']}")
            print(f"   Restante: {float(row['rows_remaining']):,.0f} linhas")

        if row['error'] and row['error'].strip():
            print(f"   Erro: {row['error'][:100]}")

    print("\n" + "=" * 60)

    # Resumo geral
    total = len(progress)
    complete = len(progress[progress['status'] == 'complete'])
    partial = len(progress[progress['status'] == 'partial'])
    error = len(progress[progress['status'].isin(['error', 'failed'])])

    print(f"Total: {total} | Completas: {complete} | Parciais: {partial} | Erros: {error}")
    print("=" * 60)


# Executar validação automática
print("Executando validação de integridade...\n")
issues = validate_migration_integrity()

print("\n")
show_migration_status()

Executando validação de integridade...

VALIDAÇÃO DE INTEGRIDADE
✅ bistage: 3,059,524 linhas OK
✅ base_cep_completa: 100,030 linhas OK
✅ tst_contratos: 3,059,524 linhas OK
❌ tst_historico_solicitacoes: esperado 15,650,000, atual 16,650,000 (diff: +1,000,000)
✅ tst_solicit_cadastradas: 100,082 linhas OK
❌ depara_cliente: esperado 47, atual 93 (diff: +46)
✅ sc5030: 50,012 linhas OK
✅ sc6030: 50,011 linhas OK
✅ sd2030: 50,012 linhas OK
✅ sf2030: 460,394 linhas OK
✅ ztx030: 50,053 linhas OK

❌ 2 TABELA(S) COM INCONSISTÊNCIA

Para corrigir, execute: fix_table_inconsistency('nome_tabela')


STATUS DA MIGRAÇÃO

🔄 ginf.TST_HISTORICO_SOLICITACOES -> tst_historico_solicitacoes
   Status: PARTIAL
   Coletado: 15,650,000 / 49,511,086
   Progresso: 31.6%
   Último ID: 3522790.0000000000
   Restante: 33,861,086 linhas

✅ bistage.TST_CONTRATOS_BI -> bistage
   Status: COMPLETE
   Coletado: 3,059,524 / 3,059,524
   Progresso: 100.0%

✅ ginf.BASE_CEP_COMPLETA -> base_cep_completa
   Status: COMPLETE
  

## Sumário da Tarefa
Criar uma célula para verificar a quantidade de linhas totalmente idênticas (duplicatas exatas) inseridas **hoje** no ClickHouse.

---



In [79]:
# Verificar duplicatas exatas inseridas hoje no ClickHouse
from datetime import datetime, timedelta

# Definir data de hoje
today = datetime.now().strftime('%Y-%m-%d')
print(f"Data de referência: {today}")
print("=" * 60)

# Obter todas as tabelas com progresso
tables_query = """
               SELECT DISTINCT ch_table, id_column
               FROM migration_progress FINAL
               WHERE status IN ('partial', 'complete')
               ORDER BY ch_table \
               """

tables_df = client.query_df(tables_query)

total_duplicates_today = 0
results = []

for idx, row in tables_df.iterrows():
    ch_tbl = row['ch_table']
    id_col = row['id_column']

    print(f"\n[{idx + 1}/{len(tables_df)}] Verificando: {ch_tbl}")

    try:
        # Query para contar duplicatas exatas inseridas hoje
        # Assumindo que existe coluna de data/timestamp para filtrar hoje
        # Se não houver, remover o filtro de data

        dup_query = f"""
        SELECT COUNT(*) as total_dups
        FROM (
            SELECT *, COUNT(*) as cnt
            FROM {ch_tbl}
            GROUP BY *
            HAVING cnt > 1
        )
        """

        result = client.query(dup_query).result_rows

        if result:
            dups_count = result[0][0]

            if dups_count > 0:
                print(f"  ❌ {dups_count:,} linha(s) duplicada(s) encontrada(s)")

                # Mostrar exemplos das duplicatas
                example_query = f"""
                SELECT *, COUNT(*) as occurrences
                FROM {ch_tbl}
                GROUP BY *
                HAVING occurrences > 1
                LIMIT 3
                """

                examples = client.query_df(example_query)
                print(f"  Exemplos (primeiras 3 duplicatas):")
                print(examples[['occurrences']].to_string(index=False))

                total_duplicates_today += dups_count
                results.append({
                    'table': ch_tbl,
                    'duplicates': dups_count
                })
            else:
                print(f"  ✅ Sem duplicatas")
        else:
            print(f"  ✅ Sem duplicatas")

    except Exception as e:
        print(f"  ⚠️ Erro: {str(e)[:100]}")

print("\n" + "=" * 60)
print("RESUMO DE DUPLICATAS TOTALMENTE IDÊNTICAS")
print("=" * 60)

if results:
    for r in results:
        print(f"  {r['table']}: {r['duplicates']:,} duplicatas")
    print(f"\nTOTAL DE DUPLICATAS: {total_duplicates_today:,}")
else:
    print("✅ Nenhuma duplicata encontrada em nenhuma tabela")

print("=" * 60)


Data de referência: 2026-02-06

[1/11] Verificando: base_cep_completa
  ❌ 50,000 linha(s) duplicada(s) encontrada(s)
  Exemplos (primeiras 3 duplicatas):
 occurrences
           2
           2
           2

[2/11] Verificando: bistage
  ❌ 12,303 linha(s) duplicada(s) encontrada(s)
  Exemplos (primeiras 3 duplicatas):
 occurrences
           2
           2
           2

[3/11] Verificando: depara_cliente
  ❌ 46 linha(s) duplicada(s) encontrada(s)
  Exemplos (primeiras 3 duplicatas):
 occurrences
           2
           2
           2

[4/11] Verificando: sc5030
  ❌ 12 linha(s) duplicada(s) encontrada(s)
  Exemplos (primeiras 3 duplicatas):
 occurrences
           2
           2
           2

[5/11] Verificando: sc6030
  ❌ 11 linha(s) duplicada(s) encontrada(s)
  Exemplos (primeiras 3 duplicatas):
 occurrences
           2
           2
           2

[6/11] Verificando: sd2030
  ❌ 12 linha(s) duplicada(s) encontrada(s)
  Exemplos (primeiras 3 duplicatas):
 occurrences
           2
       

In [80]:
`# Estimativa de Custo: ClickHouse com e sem Duplicatas

import pandas as pd

print("=" * 80)
print("ESTIMATIVA DE CUSTO - CLICKHOUSE CLOUD")
print("=" * 80)

# Obter estatísticas de todas as tabelas
tables_stats = client.query_df("""
                               SELECT oracle_table,
                                      ch_table,
                                      rows_collected as current_rows,
                                      total_rows
                               FROM migration_progress FINAL
                               WHERE status IN ('partial', 'complete')
                               ORDER BY ch_table
                               """)

# Constantes de custo ClickHouse Cloud (valores aproximados)
# Fonte: https://clickhouse.com/pricing
COST_PER_GB_STORAGE = 0.16  # USD por GB/mês (tier padrão)
COST_PER_GB_COMPUTE = 0.40  # USD por GB processado
AVG_ROW_SIZE_BYTES = 500  # Tamanho médio estimado por linha (ajustar conforme seu caso)
COMPRESSION_RATIO = 0.15  # ClickHouse comprime ~85% (varia 10-20%)

results = []
total_rows_with_dups = 0
total_rows_without_dups = 0
total_dups_found = 0

print("\nVerificando duplicatas em cada tabela...\n")

for idx, row in tables_stats.iterrows():
    ch_tbl = row['ch_table']
    current_rows = int(row['current_rows'])

    print(f"[{idx + 1}/{len(tables_stats)}] {ch_tbl}: {current_rows:,} linhas")

    try:
        # Contar linhas únicas (sem duplicatas exatas)
        unique_query = f"""
        SELECT COUNT(*) as unique_count
        FROM (
            SELECT DISTINCT *
            FROM {ch_tbl}
        )
        """

        unique_count = client.query(unique_query).result_rows[0][0]
        duplicates = current_rows - unique_count

        print(f"  Únicas: {unique_count:,} | Duplicadas: {duplicates:,}")

        results.append({
            'table': ch_tbl,
            'rows_with_dups': current_rows,
            'rows_unique': unique_count,
            'duplicates': duplicates
        })

        total_rows_with_dups += current_rows
        total_rows_without_dups += unique_count
        total_dups_found += duplicates

    except Exception as e:
        print(f"  ⚠️ Erro: {str(e)[:80]}")
        results.append({
            'table': ch_tbl,
            'rows_with_dups': current_rows,
            'rows_unique': current_rows,
            'duplicates': 0
        })
        total_rows_with_dups += current_rows
        total_rows_without_dups += current_rows

# Calcular tamanhos e custos
size_with_dups_gb = (total_rows_with_dups * AVG_ROW_SIZE_BYTES) / (1024 ** 3) * COMPRESSION_RATIO
size_without_dups_gb = (total_rows_without_dups * AVG_ROW_SIZE_BYTES) / (1024 ** 3) * COMPRESSION_RATIO
size_saved_gb = size_with_dups_gb - size_without_dups_gb

# Custos mensais de armazenamento
storage_cost_with_dups = size_with_dups_gb * COST_PER_GB_STORAGE
storage_cost_without_dups = size_without_dups_gb * COST_PER_GB_STORAGE
storage_cost_saved = size_saved_gb * COST_PER_GB_STORAGE

# Custos de computação (estimativa para 100 queries/dia)
queries_per_month = 100 * 30
data_scanned_per_query_gb = size_with_dups_gb * 0.1  # Assume 10% dos dados por query
compute_cost_with_dups = (data_scanned_per_query_gb * queries_per_month * COST_PER_GB_COMPUTE) / 1000
compute_cost_without_dups = ((size_without_dups_gb * 0.1) * queries_per_month * COST_PER_GB_COMPUTE) / 1000

# Custo total mensal
total_cost_with_dups = storage_cost_with_dups + compute_cost_with_dups
total_cost_without_dups = storage_cost_without_dups + compute_cost_without_dups
total_cost_saved = total_cost_with_dups - total_cost_without_dups

# Exibir resultados
print("\n" + "=" * 80)
print("RESUMO DE DADOS")
print("=" * 80)
print(f"Total de linhas (COM duplicatas):     {total_rows_with_dups:>20,}")
print(f"Total de linhas (SEM duplicatas):     {total_rows_without_dups:>20,}")
print(f"Duplicatas encontradas:               {total_dups_found:>20,}")
print(f"Redução de linhas:                    {(total_dups_found / total_rows_with_dups * 100):>19.2f}%")

print("\n" + "=" * 80)
print("ESTIMATIVA DE TAMANHO (após compressão ClickHouse)")
print("=" * 80)
print(f"Tamanho COM duplicatas:               {size_with_dups_gb:>19.2f} GB")
print(f"Tamanho SEM duplicatas:               {size_without_dups_gb:>19.2f} GB")
print(f"Espaço economizado:                   {size_saved_gb:>19.2f} GB")
print(f"Redução de tamanho:                   {(size_saved_gb / size_with_dups_gb * 100):>19.2f}%")

print("\n" + "=" * 80)
print("ESTIMATIVA DE CUSTO MENSAL - CLICKHOUSE CLOUD")
print("=" * 80)
print("\n1. ARMAZENAMENTO:")
print(f"   COM duplicatas:                    ${storage_cost_with_dups:>19.2f}/mês")
print(f"   SEM duplicatas:                    ${storage_cost_without_dups:>19.2f}/mês")
print(f"   Economia:                          ${storage_cost_saved:>19.2f}/mês")

print("\n2. COMPUTAÇÃO (estimativa: 100 queries/dia):")
print(f"   COM duplicatas:                    ${compute_cost_with_dups:>19.2f}/mês")
print(f"   SEM duplicatas:                    ${compute_cost_without_dups:>19.2f}/mês")
print(f"   Economia:                          ${(compute_cost_with_dups - compute_cost_without_dups):>19.2f}/mês")

print("\n3. CUSTO TOTAL MENSAL:")
print(f"   COM duplicatas:                    ${total_cost_with_dups:>19.2f}/mês")
print(f"   SEM duplicatas:                    ${total_cost_without_dups:>19.2f}/mês")
print(f"   ECONOMIA TOTAL:                    ${total_cost_saved:>19.2f}/mês")
print(f"   ECONOMIA ANUAL:                    ${(total_cost_saved * 12):>19.2f}/ano")

print("\n" + "=" * 80)
print("DETALHAMENTO POR TABELA")
print("=" * 80)

results_df = pd.DataFrame(results)
results_df['pct_dups'] = (results_df['duplicates'] / results_df['rows_with_dups'] * 100).round(2)
results_df['size_gb'] = (results_df['rows_with_dups'] * AVG_ROW_SIZE_BYTES / (1024 ** 3) * COMPRESSION_RATIO).round(3)

print(results_df.to_string(index=False))

print("\n" + "=" * 80)
print("PREMISSAS UTILIZADAS:")
print("=" * 80)
print(f"• Tamanho médio por linha:            {AVG_ROW_SIZE_BYTES} bytes")
print(f"• Taxa de compressão ClickHouse:      {(1 - COMPRESSION_RATIO) * 100:.0f}%")
print(f"• Custo armazenamento:                ${COST_PER_GB_STORAGE}/GB/mês")
print(f"• Custo computação:                   ${COST_PER_GB_COMPUTE}/GB processado")
print(f"• Queries estimadas:                  100/dia (3.000/mês)")
print("=" * 80)

print("\n💡 RECOMENDAÇÃO:")
if total_cost_saved > 10:
    print(f"   Remover duplicatas pode economizar ${total_cost_saved:.2f}/mês (${total_cost_saved * 12:.2f}/ano)")
    print("   Considere implementar deduplicação na camada SILVER.")
else:
    print(f"   Economia estimada de ${total_cost_saved:.2f}/mês é baixa.")
    print("   Manter duplicatas pode ser aceitável se houver necessidade de auditoria.")

ESTIMATIVA DE CUSTO - CLICKHOUSE CLOUD

Verificando duplicatas em cada tabela...

[1/11] base_cep_completa: 100,030 linhas
  Únicas: 50,030 | Duplicadas: 50,000
[2/11] bistage: 3,059,524 linhas
  Únicas: 3,047,221 | Duplicadas: 12,303
[3/11] depara_cliente: 47 linhas
  Únicas: 46 | Duplicadas: 1
[4/11] sc5030: 50,012 linhas
  Únicas: 50,000 | Duplicadas: 12
[5/11] sc6030: 50,011 linhas
  Únicas: 50,000 | Duplicadas: 11
[6/11] sd2030: 50,012 linhas
  Únicas: 50,000 | Duplicadas: 12
[7/11] sf2030: 460,394 linhas
  Únicas: 429,500 | Duplicadas: 30,894
[8/11] tst_contratos: 3,059,524 linhas
  Únicas: 3,047,221 | Duplicadas: 12,303
[9/11] tst_historico_solicitacoes: 17,650,000 linhas
  Únicas: 16,599,998 | Duplicadas: 1,050,002
[10/11] tst_solicit_cadastradas: 100,082 linhas
  Únicas: 50,097 | Duplicadas: 49,985
[11/11] ztx030: 50,053 linhas
  Únicas: 50,052 | Duplicadas: 1

RESUMO DE DADOS
Total de linhas (COM duplicatas):               24,629,689
Total de linhas (SEM duplicatas):         

In [84]:
from datetime import datetime
from pyspark.sql import functions as F
import pandas as pd

print("=" * 80)
print("ANÁLISE DE VOLUMETRIA - TABELAS ORACLE")
print("=" * 80)

# Definir todas as tabelas organizadas por categoria
tables_to_analyze = {
    "Core / Contratos / Clientes": [
        "CN9030", "CN1030", "CNB030", "SA1030", "SA3030",
        "SB1030", "ZB3030", "SZJ030", "SZH030", "SZU030",
        "SZV030", "SZW030", "ZE8030", "ZT1030", "ZTX030",
        "ZAA030", "ZA1030", "ZA3030"
    ],
    "ERP": [
        "ERP_AGREEMENT", "ERP_PRODUCT", "ERP_PRODUCT_ITEM", "ERP_VEHICLE"
    ],
    "SC - Workflow / Requisição / Instalação": [
        "SC_REQUISITION", "SC_REQUISITION_QUEUE", "SC_REQUISITION_HISTORY",
        "SC_REQUISITION_STATUS", "SC_REQ_FILE", "SC_TASK", "SC_GROUP",
        "SC_RESULT_CODE", "SC_RESERVE", "SC_RESERVE_LOCATION", "SC_LOCATION",
        "SC_WAREHOUSE", "SC_CITY", "SC_STATE", "SC_ROLE"
    ],
    "Views / Integrações": [
        "VW_USER", "SC_TECHNICAL_REGISTER", "SC_WEBSERVICE_REQUISITION"
    ],
    "Fiscal / Financeiro": [
        "SF2030", "SD2030", "SC5030", "SC6030", "SE4030"
    ],
    "Bases Auxiliares": [
        "CEPREG", "BASE_REGIONAL", "TAB_CIDADE_DELITO_SP_CAP"
    ]
}

# Mapear schemas corretos
schema_map = {
    "Core / Contratos / Clientes": "siga",
    "Fiscal / Financeiro": "siga",
    "SC - Workflow / Requisição / Instalação": "scot",
    "Views / Integrações": "scot",
    "Bases Auxiliares": "ginf",
    "ERP": "erp"
}

# Acumular resultados
all_results = []

start_time = datetime.now()
total_tables = sum(len(tables) for tables in tables_to_analyze.values())
processed = 0

for category, tables in tables_to_analyze.items():
    print(f"\n{'=' * 80}")
    print(f"CATEGORIA: {category}")
    print(f"{'=' * 80}")

    category_total_rows = 0
    category_total_size_mb = 0
    category_success = 0
    category_failed = 0

    schema = schema_map.get(category, "ginf")

    for table_name in tables:
        processed += 1
        pct = (processed / total_tables) * 100
        full_table = f"{schema}.{table_name}"
        print(f"\n[{processed}/{total_tables}] ({pct:.1f}%) Analisando: {full_table}")

        try:
            t0 = datetime.now()

            # Contar linhas - query corrigida
            count_query = f"(SELECT COUNT(*) as CNT FROM {schema}.{table_name}) tmp"
            df_count = spark.read.format("jdbc") \
                .options(**jdbc_opts) \
                .option("dbtable", count_query) \
                .load()

            row_count = int(df_count.collect()[0]['CNT'])

            # Estimar tamanho
            if row_count > 0:
                sample_size = min(1000, row_count)
                sample_query = f"(SELECT * FROM {schema}.{table_name} WHERE ROWNUM <= {sample_size}) tmp"
                df_sample = spark.read.format("jdbc") \
                    .options(**jdbc_opts) \
                    .option("dbtable", sample_query) \
                    .load()

                num_cols = len(df_sample.columns)
                avg_row_size = num_cols * 100
                estimated_size_mb = (row_count * avg_row_size) / (1024 * 1024)
            else:
                num_cols = 0
                estimated_size_mb = 0

            duration = (datetime.now() - t0).total_seconds()

            print(f"  ✅ Linhas: {row_count:,}")
            print(f"  📊 Colunas: {num_cols}")
            print(f"  💾 Tamanho estimado: {estimated_size_mb:,.2f} MB")
            print(f"  ⏱️  Tempo: {duration:.2f}s")

            all_results.append({
                'categoria': category,
                'tabela': full_table,
                'linhas': row_count,
                'colunas': num_cols,
                'tamanho_mb': estimated_size_mb,
                'status': 'OK'
            })

            category_total_rows += row_count
            category_total_size_mb += estimated_size_mb
            category_success += 1

        except Exception as e:
            error_msg = str(e)[:100]
            print(f"  ❌ ERRO: {error_msg}")

            all_results.append({
                'categoria': category,
                'tabela': full_table,
                'linhas': 0,
                'colunas': 0,
                'tamanho_mb': 0,
                'status': f'ERRO: {error_msg}'
            })

            category_failed += 1

    print(f"\n{'=' * 80}")
    print(f"RESUMO - {category}")
    print(f"{'=' * 80}")
    print(f"Total de linhas: {category_total_rows:,}")
    print(f"Tamanho estimado: {category_total_size_mb:,.2f} MB ({category_total_size_mb / 1024:.2f} GB)")
    print(f"Sucesso: {category_success}/{len(tables)}")
    if category_failed > 0:
        print(f"Falhas: {category_failed}")

total_duration = (datetime.now() - start_time).total_seconds()

results_df = pd.DataFrame(all_results)

print(f"\n{'=' * 80}")
print("RESUMO GERAL - TODAS AS CATEGORIAS")
print(f"{'=' * 80}")

total_all_rows = results_df['linhas'].sum()
total_all_size_mb = results_df['tamanho_mb'].sum()
total_success = len(results_df[results_df['status'] == 'OK'])
total_failed = len(results_df[results_df['status'] != 'OK'])

print(f"Total de tabelas analisadas: {total_tables}")
print(f"Sucesso: {total_success}")
print(f"Falhas: {total_failed}")
print(f"\nTotal de linhas: {total_all_rows:,}")
print(f"Tamanho total estimado: {total_all_size_mb:,.2f} MB ({total_all_size_mb / 1024:.2f} GB)")
print(f"Tempo total: {total_duration:.2f}s")

print(f"\n{'=' * 80}")
print("TOP 10 TABELAS POR VOLUMETRIA")
print(f"{'=' * 80}")

top_tables = results_df[results_df['status'] == 'OK'].nlargest(10, 'linhas')
for idx, row in top_tables.iterrows():
    print(f"{row['tabela']:50s} {row['linhas']:>15,} linhas  {row['tamanho_mb']:>10,.2f} MB")

print(f"\n{'=' * 80}")
print("VOLUMETRIA POR CATEGORIA")
print(f"{'=' * 80}")

category_stats = results_df.groupby('categoria').agg({
    'linhas': 'sum',
    'tamanho_mb': 'sum'
}).sort_values('linhas', ascending=False)

for cat, stats in category_stats.iterrows():
    print(f"\n{cat}")
    print(f"  Linhas: {stats['linhas']:,}")
    print(f"  Tamanho: {stats['tamanho_mb']:,.2f} MB ({stats['tamanho_mb'] / 1024:.2f} GB)")

failed_tables = results_df[results_df['status'] != 'OK']
if len(failed_tables) > 0:
    print(f"\n{'=' * 80}")
    print(f"TABELAS COM ERRO ({len(failed_tables)})")
    print(f"{'=' * 80}")
    for idx, row in failed_tables.iterrows():
        print(f"  ❌ {row['tabela']}")
        print(f"     {row['status']}")

print(f"\n{'=' * 80}")

results_df


ANÁLISE DE VOLUMETRIA - TABELAS ORACLE

CATEGORIA: Core / Contratos / Clientes

[1/48] (2.1%) Analisando: siga.CN9030
  ✅ Linhas: 276,206
  📊 Colunas: 170
  💾 Tamanho estimado: 4,477.98 MB
  ⏱️  Tempo: 1.15s

[2/48] (4.2%) Analisando: siga.CN1030
  ✅ Linhas: 16
  📊 Colunas: 36
  💾 Tamanho estimado: 0.05 MB
  ⏱️  Tempo: 0.82s

[3/48] (6.2%) Analisando: siga.CNB030
  ✅ Linhas: 3,719,489
  📊 Colunas: 94
  💾 Tamanho estimado: 33,343.50 MB
  ⏱️  Tempo: 1.24s

[4/48] (8.3%) Analisando: siga.SA1030
  ✅ Linhas: 906,445
  📊 Colunas: 318
  💾 Tamanho estimado: 27,489.62 MB
  ⏱️  Tempo: 0.99s

[5/48] (10.4%) Analisando: siga.SA3030
  ✅ Linhas: 3,104
  📊 Colunas: 111
  💾 Tamanho estimado: 32.86 MB
  ⏱️  Tempo: 0.99s

[6/48] (12.5%) Analisando: siga.SB1030
  ✅ Linhas: 9,314
  📊 Colunas: 324
  💾 Tamanho estimado: 287.79 MB
  ⏱️  Tempo: 1.14s

[7/48] (14.6%) Analisando: siga.ZB3030
  ✅ Linhas: 62
  📊 Colunas: 8
  💾 Tamanho estimado: 0.05 MB
  ⏱️  Tempo: 0.82s

[8/48] (16.7%) Analisando: siga.SZJ030
  

,categoria,tabela,linhas,colunas,tamanho_mb,status
0,Core / Contratos / Clientes,siga.CN9030,276206,170,4477.979660,OK
1,Core / Contratos / Clientes,siga.CN1030,16,36,0.054932,OK
2,Core / Contratos / Clientes,siga.CNB030,3719489,94,33343.502617,OK
3,Core / Contratos / Clientes,siga.SA1030,906445,318,27489.615440,OK
4,Core / Contratos / Clientes,siga.SA3030,3104,111,32.858276,OK
5,Core / Contratos / Clientes,siga.SB1030,9314,324,287.793732,OK
6,Core / Contratos / Clientes,siga.ZB3030,62,8,0.047302,OK
7,Core / Contratos / Clientes,siga.SZJ030,73,8,0.055695,OK
8,Core / Contratos / Clientes,siga.SZH030,17,5,0.008106,OK
9,Core / Contratos / Clientes,siga.SZU030,1665204,37,5875.830460,OK


In [85]:
# Retornar DataFrame com resultados da análise de volumetria
results_df


,categoria,tabela,linhas,colunas,tamanho_mb,status
0,Core / Contratos / Clientes,siga.CN9030,276206,170,4477.979660,OK
1,Core / Contratos / Clientes,siga.CN1030,16,36,0.054932,OK
2,Core / Contratos / Clientes,siga.CNB030,3719489,94,33343.502617,OK
3,Core / Contratos / Clientes,siga.SA1030,906445,318,27489.615440,OK
4,Core / Contratos / Clientes,siga.SA3030,3104,111,32.858276,OK
5,Core / Contratos / Clientes,siga.SB1030,9314,324,287.793732,OK
6,Core / Contratos / Clientes,siga.ZB3030,62,8,0.047302,OK
7,Core / Contratos / Clientes,siga.SZJ030,73,8,0.055695,OK
8,Core / Contratos / Clientes,siga.SZH030,17,5,0.008106,OK
9,Core / Contratos / Clientes,siga.SZU030,1665204,37,5875.830460,OK
